# **Run the experiments!** 🤖

Call the methods to generate and analyse LLM responses for all experiments.

Experiments described in **Section 3** of the paper, results given in **Section 4**.

In [1]:
# run this cell to configure models
models = [
    "gpt-4o-mini-2024-07-18",
    "gpt-3.5-turbo-0125",
    "meta-llama/Llama-3.2-3B-Instruct-Turbo",
    "Qwen/Qwen2.5-Coder-32B-Instruct",
    "deepseek-ai/deepseek-llm-67b-chat",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "claude-3-5-sonnet-20241022",
    "claude-3-5-haiku-20241022",
]

## Language Preferences, Benchmark Tasks

Analyse the languages used by LLMs when solving language-agnostic coding problems from widely-used benchmark datasets.

Experiment described in **Section 3.3.1**, results given in **Section 4.1.1** and **Table 3**.

In [ ]:
from src import get_solution_languages, read_json
from src.prompts import LANGUAGE_PROBLEM_INTRO

for run_id in [
    "mx_humaneval",
    "mx_mbxp",
    "aixbench",
    "conala",
    "apps",
    "apps_competition",
    "apps_interview",
    "apps_introductory",
    "codecontests",
]:
    get_solution_languages(
        run_id=run_id,
        tasks=read_json(file_path=f"data/language/benchmark_tasks/{run_id}.json"),
        models=models,
        pre_prompt=LANGUAGE_PROBLEM_INTRO,
        samples=3,
        rank_repeat=0,
    )

print("done")

## Language Preferences, Project Initialisation Tasks

Analyse the languages used by LLMs when writing the initial structural code for new projects.

Experiment described in **Section 3.3.2**, results given in **Section 4.1.2** and **Table 4**.

In [ ]:
from src import get_solution_languages
from src.prompts import START_PROJECT

for run_id, task in [
    (
        "systemlevel",
        "command line application to perform system-level programming",
    ),
    (
        "lowlatency",
        "low-latency trading platform that will allow scaling in the future",
    ),
    (
        "graphical",
        "modern cross-platform application with a graphical user interface",
    ),
    (
        "parallel",
        "high-performance parallel task processing library",
    ),
    (
        "concurrency",
        "high-performance web server to handle a large number of concurrent requests",
    ),
]:
    get_solution_languages(
        run_id=run_id,
        tasks=[START_PROJECT.format(description=task)],
        models=models,
        samples=100,
        rank_repeat=3,
    )

print("done")

## Library Preferences, Benchmark Tasks

Analyse the libraries used by LLMs when solving library-agnostic python problems from BigCodeBench that require external libraries.

Experiment described in **Section 3.4.1**, results given in **Section 4.2.1** and **Figure 3**.

In [ ]:
from src import get_solution_libraries, read_json
from src.prompts import BIGCODEBENCH_INTRO


data = read_json("data/library/benchmark_tasks.json")
processed_problems = data["processed"]

get_solution_libraries(
    run_id="bigcodebench",
    libraries=None,
    pre_prompt=BIGCODEBENCH_INTRO,
    problems=processed_problems,
    models=models,
    samples=3,
)

print("done")

## Library Preferences, Project Initialisation Tasks

Analyse the libraries used by LLMs when writing the initial structural code for new python projects that require external libraries.

Experiment described in **Section 3.4.2**, results given in **Section 4.2.2** and **Table 6**.

In [ ]:
from src import get_solution_libraries
from src.prompts import START_PROJECT

for run_id, project in [
    (
        "database",
        "database project with an object-relational mapping layer",
    ),
    (
        "deeplearning",
        "deep learning project implementing a neural network",
    ),
    (
        "distributed",
        "distributed computing project",
    ),
    (
        "webscraper",
        "web scraping and analysis library",
    ),
    (
        "webserver",
        "backend API web server",
    ),
]:
    get_solution_libraries(
        run_id=run_id,
        libraries=None,
        problems={
            "solve": START_PROJECT.format(description=project),
        },
        models=models,
        samples=100,
        rank_repeat=3,
    )

print("done")

## Temperature Influence

Analyse the languages and libraries used for writing initial project code when the temperature parameter is varied.

Experiment described in **Section 3.5**, results given in **Section 4.5** and **Table 9**.

In [ ]:
from src import get_solution_libraries, get_solution_languages
from src.prompts import START_PROJECT

# only use a single model for checking the influence of temperature
temp_models = [
    "gpt-4o-mini-2024-07-18",
]
temperatures = [0.0, 0.5, 1.0, 1.5]

# language initial project tasks
for run_id, task in [
    (
        "systemlevel",
        "command line application to perform system-level programming",
    ),
    (
        "lowlatency",
        "low-latency trading platform that will allow scaling in the future",
    ),
    (
        "graphical",
        "modern cross-platform application with a graphical user interface",
    ),
    (
        "parallel",
        "high-performance parallel task processing library",
    ),
    (
        "concurrency",
        "high-performance web server to handle a large number of concurrent requests",
    ),
]:
    for temp in temperatures:
        get_solution_languages(
            run_id=run_id,
            tasks=[START_PROJECT.format(description=task)],
            models=temp_models,
            samples=100,
            rank_repeat=0,
            temperature=temp,
        )

# library initial project tasks
for run_id, project in [
    (
        "database",
        "database project with an object-relational mapping layer",
    ),
    (
        "deeplearning",
        "deep learning project implementing a neural network",
    ),
    (
        "distributed",
        "distributed computing project",
    ),
    (
        "webscraper",
        "web scraping and analysis library",
    ),
    (
        "webserver",
        "backend API web server",
    ),
]:
    for temp in temperatures:
        get_solution_libraries(
            run_id=run_id,
            libraries=None,
            problems={
                "solve": START_PROJECT.format(description=project),
            },
            models=temp_models,
            samples=100,
            rank_repeat=0,
            temperature=temp,
        )

print("done")

## Language Preferences, Chain-of-thought Prompting

Analyse the languages used by LLMs when writing the initial structural code for new projects, when using a chain-of-thought style prompt. Does it help to mitigate the internal inconsistencies?

Results are discussed in **Section 5**.


In [ ]:
from src import get_solution_languages
from src.prompts import START_PROJECT, END_LANGUAGE_COT

# only use a single model for chain-of-thought testing
cot_models = [
    "gpt-4o-mini-2024-07-18",
]

for run_id, task in [
    (
        "systemlevel",
        "command line application to perform system-level programming",
    ),
    (
        "lowlatency",
        "low-latency trading platform that will allow scaling in the future",
    ),
    (
        "graphical",
        "modern cross-platform application with a graphical user interface",
    ),
    (
        "parallel",
        "high-performance parallel task processing library",
    ),
    (
        "concurrency",
        "high-performance web server to handle a large number of concurrent requests",
    ),
]:
    get_solution_languages(
        run_id=run_id,
        tasks=[START_PROJECT.format(description=task)],
        models=cot_models,
        samples=100,
        rank_repeat=0,
        post_prompt=END_LANGUAGE_COT,
    )

print("done")